*Disclaimer - the following data is synthetic AI generated for practice purposes 

**Initial Steps**
First, Pandas is imported and the CSV file is loaded. An initial inspection of the table is done to understand the shape and quality of the data, data types, and to get a quick sense of where inconsistencies might be hiding. 

In [6]:
import pandas as pd

df = pd.read_csv("messy_claims_data.csv")
df.head(10)

,patient_id,patient_name,claim_date,diagnosis_code,claim_amount,provider_id,claim_status
0,P0021,Maria Smith,2025-05-21,E78.5,740.71,NaN,Paid
1,P0019,David Smith,01/16/2025,E78.5,1201.67,NaN,paid
2,P0023,Carlos Wilson,2025-08-03,I10,2966.87,PR01,paid
3,P0023,David Davis,2025-05-23,E78.5,4788.20,PR01,Denied
4,P0004,Patricia Davis,05/16/2025,I10,2704.33,PR04,Paid
5,P0018,Michael Lopez,2025-07-05,I10,276.83,PR03,Paid
6,P0028,Linda Garcia,14-Jul-2025,I10,3196.64,PR03,PAID
7,P0007,Carlos Brown,2025-02-06,E78.5,858.84,NaN,paid
8,P0022,Patricia Smith,04/28/2025,NaN,NaN,NaN,denied
9,P0011,Linda Miller,07-22-25,J45.909,1361.16,NaN,Pending


In [7]:
df.shape

(65, 7)

In [8]:
df.dtypes

patient_id            str
patient_name          str
claim_date            str
diagnosis_code        str
claim_amount      float64
provider_id           str
claim_status          str
dtype: object

**Removing Duplicate Records**
Five rows were found to be exact duplicates. These were removed to prevent duplicate entries from being counted in the the final summary.

In [9]:
df.duplicated().sum()

np.int64(5)

In [10]:
df = df.drop_duplicates()
df.shape

(60, 7)

**Standaridzing Claim Date**
The claim_date column has mixed formats being used to report the dates, which may indicate data was merges from different source systems. Used format="mixed" to parse each row in a single consistenet datetime type. 

In [11]:
df["claim_date"] = pd.to_datetime(df["claim_date"], format="mixed")
df["claim_date"].head(10)
df.dtypes

patient_id                   str
patient_name                 str
claim_date        datetime64[us]
diagnosis_code               str
claim_amount             float64
provider_id                  str
claim_status                 str
dtype: object

In [12]:
df["claim_date"].head(10)

0   2025-05-21
1   2025-01-16
2   2025-08-03
3   2025-05-23
4   2025-05-16
5   2025-07-05
6   2025-07-14
7   2025-02-06
8   2025-04-28
9   2025-07-22
Name: claim_date, dtype: datetime64[us]

**Standaridzing Claim Status**
The claim_date column has mixed formats being used to report the dates, which may indicate data was merges from different source systems. Used format="mixed" to parse each row in a single consistenet datetime type. 

In [13]:
df["claim_status"] = df["claim_status"].str.strip().str.lower()
df["claim_status"].value_counts()

claim_status
paid       38
denied     14
pending     8
Name: count, dtype: int64

In [14]:
df["diagnosis_code"] = df["diagnosis_code"].str.strip().str.upper()
df["diagnosis_code"].value_counts(dropna=False)

diagnosis_code
I10        17
E78.5      14
E11.9      14
N18.3       6
J45.909     5
NaN         4
Name: count, dtype: int64

In [15]:
df["diagnosis_code"] = df["diagnosis_code"].fillna("Unknown")
df["diagnosis_code"].value_counts(dropna=False)

diagnosis_code
I10        17
E78.5      14
E11.9      14
N18.3       6
J45.909     5
Unknown     4
Name: count, dtype: int64

In [18]:
df["claim_amount"].isna().sum()
print(df["claim_amount"].isna().sum())

5


In [25]:
df["amount_missing"] = df["claim_amount"].isna()
df["provider_id"] = df["provider_id"].fillna("Unknown")

In [26]:
df["provider_id"].isna().sum()
print(df["provider_id"].isna().sum())

0


In [27]:
df["provider_id"] = df["provider_id"].fillna("Unknown")

In [28]:
df["provider_id"].value_counts(dropna=False)
df["amount_missing"].sum()

np.int64(5)

In [29]:
providers = pd.read_csv("provider_lookup.csv")
providers.head()

,provider_id,provider_name,specialty
0,PR01,Dr. Amina Yusuf,Primary Care
1,PR02,Dr. Ben Cole,Cardiology
2,PR03,Dr. Priya Nair,Endocrinology
3,PR04,Dr. Sam Osei,Primary Care


In [30]:
df_joined = df.merge(providers, on="provider_id", how="left")
df_joined.head(10)

,patient_id,patient_name,claim_date,diagnosis_code,claim_amount,provider_id,claim_status,amount_missing,provider_name,specialty
0,P0021,Maria Smith,2025-05-21,E78.5,740.71,Unknown,paid,False,NaN,NaN
1,P0019,David Smith,2025-01-16,E78.5,1201.67,Unknown,paid,False,NaN,NaN
2,P0023,Carlos Wilson,2025-08-03,I10,2966.87,PR01,paid,False,Dr. Amina Yusuf,Primary Care
3,P0023,David Davis,2025-05-23,E78.5,4788.20,PR01,denied,False,Dr. Amina Yusuf,Primary Care
4,P0004,Patricia Davis,2025-05-16,I10,2704.33,PR04,paid,False,Dr. Sam Osei,Primary Care
5,P0018,Michael Lopez,2025-07-05,I10,276.83,PR03,paid,False,Dr. Priya Nair,Endocrinology
6,P0028,Linda Garcia,2025-07-14,I10,3196.64,PR03,paid,False,Dr. Priya Nair,Endocrinology
7,P0007,Carlos Brown,2025-02-06,E78.5,858.84,Unknown,paid,False,NaN,NaN
8,P0022,Patricia Smith,2025-04-28,Unknown,NaN,Unknown,denied,True,NaN,NaN
9,P0011,Linda Miller,2025-07-22,J45.909,1361.16,Unknown,pending,False,NaN,NaN


In [31]:
df_joined[df_joined["provider_id"] == "Unknown"]

,patient_id,patient_name,claim_date,diagnosis_code,claim_amount,provider_id,claim_status,amount_missing,provider_name,specialty
0,P0021,Maria Smith,2025-05-21,E78.5,740.71,Unknown,paid,False,NaN,NaN
1,P0019,David Smith,2025-01-16,E78.5,1201.67,Unknown,paid,False,NaN,NaN
7,P0007,Carlos Brown,2025-02-06,E78.5,858.84,Unknown,paid,False,NaN,NaN
8,P0022,Patricia Smith,2025-04-28,Unknown,NaN,Unknown,denied,True,NaN,NaN
9,P0011,Linda Miller,2025-07-22,J45.909,1361.16,Unknown,pending,False,NaN,NaN
10,P0009,Jennifer Lopez,2025-08-08,Unknown,1135.64,Unknown,denied,False,NaN,NaN
12,P0013,Susan Miller,2025-09-28,E11.9,3417.40,Unknown,paid,False,NaN,NaN
14,P0025,Robert Wilson,2025-02-24,E78.5,806.55,Unknown,pending,False,NaN,NaN
16,P0002,Linda Lopez,2025-02-10,I10,4089.32,Unknown,paid,False,NaN,NaN
20,P0003,Thomas Smith,2025-06-19,E78.5,1428.46,Unknown,paid,False,NaN,NaN


In [32]:
print("Total claims:", df_joined.shape[0])
print("Total claim amount:", df_joined["claim_amount"].sum())
print("Average claim amount:", df_joined["claim_amount"].mean())

Total claims: 60
Total claim amount: 128445.95000000001
Average claim amount: 2335.3809090909094


In [33]:
df_joined.groupby("claim_status")["claim_amount"].agg(["count", "sum"])

,count,sum
claim_status,,
denied,12,38620.05
paid,36,81667.96
pending,7,8157.94


In [34]:
df_joined.groupby("specialty")["claim_amount"].agg(["count", "sum"])

,count,sum
specialty,,
Cardiology,8,19897.09
Endocrinology,10,23350.25
Primary Care,21,53309.87
